In [1]:
# ============================================================
# TRANSFORM RACES — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [ ]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 22:06:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [3]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/races/races.parquet"
silver_folder = f"{SILVER_ROOT}/races"
silver_file = f"{silver_folder}/races_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/races/races.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/races/races_silver.parquet


In [4]:
# -----------------------------------------
# 3. Read Bronze Races
# -----------------------------------------
read_df = spark.read.parquet(bronze_file)
print("✔ Bronze races loaded")
read_df.show(5)

✔ Bronze races loaded
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-08 21:28:...|/Users/manoharaza...|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-08 21:28:...|/Users/manoharaza...|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-08 21:28:...|/Users/manoharaza...|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-08 21:28:...|/Users/manoharaza...|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|2026-06-08 21:28:...|/Users/manoharaza...|
+------+--

In [5]:
# -----------------------------------------
# 4. Select required columns (drop url)
# -----------------------------------------
races_selected_df = read_df.select(
    F.col("season"),
    F.col("round"),
    F.col("circuitId"),
    F.col("date"),
    F.col("raceName"),
    F.col("ingest_timestamp"),
    F.col("source_file")
)

In [7]:
# -----------------------------------------
# 5. Standardize + rename columns
# -----------------------------------------
races_renamed_df = (
    races_selected_df
        .withColumnRenamed("season", "year")
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("date", "race_date")
        .withColumnRenamed("raceName", "race_name")
)

print("✔ Races renamed preview")
races_renamed_df.show(5)

✔ Races renamed preview
+----+-----+------------+----------+------------------+--------------------+--------------------+
|year|round|  circuit_id| race_date|         race_name|    ingest_timestamp|         source_file|
+----+-----+------------+----------+------------------+--------------------+--------------------+
|1950|    1| silverstone|1950-05-13|british grand prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    2|      monaco|1950-05-21| monaco grand prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    3|indianapolis|1950-05-30|  indianapolis 500|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    4|  bremgarten|1950-06-04|  swiss grand prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    5|         spa|1950-06-18|belgian grand prix|2026-06-08 21:28:...|/Users/manoharaza...|
+----+-----+------------+----------+------------------+--------------------+--------------------+
only showing top 5 rows


In [8]:
# -----------------------------------------
# 6. Remove duplicates (year, round)
# -----------------------------------------
races_distinct_df = races_renamed_df.dropDuplicates(["year", "round"])

In [9]:
# -----------------------------------------
# 7. Title Case race_name
# -----------------------------------------
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

print("✔ Races Silver preview")
races_final_df.show(5)

✔ Races Silver preview
+----+-----+------------+----------+------------------+--------------------+--------------------+
|year|round|  circuit_id| race_date|         race_name|    ingest_timestamp|         source_file|
+----+-----+------------+----------+------------------+--------------------+--------------------+
|1950|    1| silverstone|1950-05-13|British Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    2|      monaco|1950-05-21| Monaco Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    3|indianapolis|1950-05-30|  Indianapolis 500|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    4|  bremgarten|1950-06-04|  Swiss Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    5|         spa|1950-06-18|Belgian Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
+----+-----+------------+----------+------------------+--------------------+--------------------+
only showing top 5 rows


In [10]:
# -----------------------------------------
# 8. Write Silver output (Option C)
# -----------------------------------------
(
    races_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"✔ Races Silver written to: {silver_file}")

✔ Races Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/races/races_silver.parquet


In [11]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5)

+----+-----+------------+----------+------------------+--------------------+--------------------+
|year|round|  circuit_id| race_date|         race_name|    ingest_timestamp|         source_file|
+----+-----+------------+----------+------------------+--------------------+--------------------+
|1950|    1| silverstone|1950-05-13|British Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    2|      monaco|1950-05-21| Monaco Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    3|indianapolis|1950-05-30|  Indianapolis 500|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    4|  bremgarten|1950-06-04|  Swiss Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
|1950|    5|         spa|1950-06-18|Belgian Grand Prix|2026-06-08 21:28:...|/Users/manoharaza...|
+----+-----+------------+----------+------------------+--------------------+--------------------+
only showing top 5 rows
